# 02 — Explore the weather text and the radar images

**Goal (CLAUDE.md, Phase 1, steps 1.3 and 1.4):** understand the weather sources before joining them to flights.
- **Part A — weather text:** METAR (the weather **now**) and TAF (the weather **forecast**).
- **Part B — radar:** the national image, the crop around an airport, pixel value → storm intensity.

Rules used in this notebook:
- Raw files are read from `data_dir` (outside the repo). **Nothing is written to disk and nothing is downloaded.**
- **Every time is UTC:** METAR `valid`, TAF `valid` / `fx_valid`, radar file names and crop keys. Timestamps are parsed as **UTC-aware** values
  (they print with `+00:00`), so a local time can never be mixed in by mistake.
- Only the airports and dates needed are read (mostly one month, July 2025, like notebook 01).
- Weather observed **after** the prediction moment is only **looked at** here. It is never a model input.

---
# PART A — Weather text (METAR and TAF)

## A0. Setup

**What:** import the libraries and read `config.yaml`: data folder, the 10 hubs, the radar settings and the prediction offset (2 h).
**Why:** paths and settings are never written by hand in the code. `airportsdata` gives each airport's ICAO code, position and time zone.

In [ ]:
import re
from pathlib import Path

import airportsdata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from matplotlib.colors import ListedColormap
from metar import Metar
from PIL import Image

# The notebook runs from notebooks/, config.yaml sits one folder up (the repo root).
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = yaml.safe_load((REPO / "config.yaml").read_text(encoding="utf-8"))

HUBS = cfg["hubs"]
RAW = Path(cfg["data_dir"]) / "raw"
PRODUCT = cfg["radar"]["product"]                      # n0q
CROP_PX = cfg["radar"]["crop_px"]                      # 384
EXPECTED_WLD = cfg["radar"]["expected_wld"]
EXPECTED_SHAPE = tuple(cfg["radar"]["expected_shape"])
T_PRED_H = cfg["t_pred_hours_before_departure"]        # 2
# One entry per airport: AIRPORTS["ATL"] = {"icao": "KATL", "lat": ..., "lon": ..., "tz": ...}
AIRPORTS = airportsdata.load("IATA")

MONTH_START, MONTH_END = "2025-07-01", "2025-08-01"    # July 2025 (the end is not included)
pd.set_option("display.max_colwidth", 200)             # show the raw reports in full

print("hubs:", HUBS)
print("raw folder:", RAW, "| exists:", RAW.exists())
print("crop:", CROP_PX, "px | T_pred = departure -", T_PRED_H, "h")

**What:** three small helpers.
- `utc(...)` turns a text like `"2025-07-15 18:30"` into a UTC-aware timestamp.
- `read_metar(iata, start, end)` reads **one** airport's METAR file and keeps `start <= valid < end`.
- `read_taf(icao, start, end)` does the same for a TAF file, on the **issue time** (`valid`).

**Why:** one file per airport, a few MB each: we open only the airports we need and keep only the dates we need.
The METAR files are named with the IATA code (`ATL.csv`), the TAF files with the ICAO code (`KATL.csv`).
`is_amendment` is read with an explicit type (`boolean` = True / False / empty), otherwise pandas complains about mixed types.

In [ ]:
def utc(when) -> pd.Timestamp:
    """Text, date or timestamp -> UTC-aware timestamp ('2025-07-15' -> 2025-07-15 00:00+00:00)."""
    ts = pd.Timestamp(when)
    # no time zone yet: declare it UTC. Already has one: convert it to UTC.
    return ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")


def read_metar(iata: str, start, end) -> pd.DataFrame:
    """METAR reports of one airport with start <= valid < end (UTC), sorted by time."""
    df = pd.read_csv(RAW / "metar" / f"{iata}.csv")
    df["valid"] = pd.to_datetime(df["valid"], utc=True)      # the file is in UTC: say so
    keep = (df["valid"] >= utc(start)) & (df["valid"] < utc(end))
    return df[keep].sort_values("valid").reset_index(drop=True)


def read_taf(icao: str, start, end) -> pd.DataFrame:
    """TAF forecast groups of one airport, for TAFs ISSUED in start <= valid < end (UTC)."""
    df = pd.read_csv(RAW / "taf" / f"{icao}.csv", dtype={"is_amendment": "boolean"})
    for col in ["valid", "fx_valid", "fx_valid_end"]:
        df[col] = pd.to_datetime(df[col], utc=True)
    keep = (df["valid"] >= utc(start)) & (df["valid"] < utc(end))
    return df[keep].reset_index(drop=True)                   # file order is kept


sample = read_metar("ATL", "2025-07-15", "2025-07-16")
print("type of `valid`:", sample["valid"].dtype)
sample.head(3)

## A1. One day of METAR at three airports

**What:** all METAR reports of ATL, JFK and MIA on 2025-07-15 (UTC day), counted per airport.
**Why:** a METAR is the airport's weather **observation**. We must know how often it comes before we write the rule "last report at or before the prediction moment".

Two kinds of report exist:
- the **hourly** report, always at the same minute for a given airport (just before the full hour);
- the **special** report (`SPECI`), sent between two hourly reports when the weather changes fast (storm starts, visibility drops, wind shifts).

In these IEM files the words `METAR` / `SPECI` are **not written**. We recognise a special report by its **minute**: it is not the airport's usual minute.

In [ ]:
DAY, NEXT_DAY = "2025-07-15", "2025-07-16"
THREE = ["ATL", "JFK", "MIA"]

day = pd.concat([read_metar(a, DAY, NEXT_DAY) for a in THREE], ignore_index=True)
day["minute"] = day["valid"].dt.minute
# The minute seen most often at an airport = the minute of its routine hourly report.
routine_minute = day.groupby("station")["minute"].agg(lambda m: m.mode().iloc[0])
is_hourly = day["minute"] == day["station"].map(routine_minute)
day["kind"] = np.where(is_hourly, "hourly", "special")

print("METAR rows read:", len(day))
per_kind = day.groupby(["station", "kind"]).size().unstack(fill_value=0)
per_kind["routine minute"] = routine_minute
per_kind

**What:** number of reports per airport and per UTC hour.
**Why:** an hour with 2 or more reports is an hour where something changed (a special was sent).

In [ ]:
per_hour = day.groupby(["station", day["valid"].dt.hour]).size().unstack(fill_value=0)
per_hour.columns.name = "UTC hour"
per_hour

**What:** the special reports of that day, in full.
**Why:** read them: what changed? (Look for `TS`, a lower visibility, a stronger wind, or `TSB..` / `TSE..` in the remarks.)

In [ ]:
day.loc[day["kind"] == "special", ["station", "valid", "metar"]]

**What:** two things to look for in the text.
- `COR` after the time = a **corrected** report (the first version had an error).
- `$` at the very end = the automatic station says "**I need maintenance**" (a sensor may be unreliable). The report is still valid.

Counted on the day and on the whole month of July (more examples).

**Why:** Silver must know what to do with both. A correction replaces a report; a `$` only lowers our trust in it.

In [ ]:
def add_report_flags(df: pd.DataFrame) -> pd.DataFrame:
    """Add two True/False columns: `is_cor` (corrected) and `maintenance` (ends with $)."""
    df = df.copy()
    # \b = word border: finds the separate word COR, not the letters inside another word
    df["is_cor"] = df["metar"].str.contains(r"\bCOR\b")
    df["maintenance"] = df["metar"].str.rstrip().str.endswith("$")
    return df


day = add_report_flags(day)
month3 = add_report_flags(
    pd.concat([read_metar(a, MONTH_START, MONTH_END) for a in THREE], ignore_index=True))

counts = pd.DataFrame({
    "reports (day)": day.groupby("station").size(),
    "COR (day)": day.groupby("station")["is_cor"].sum(),
    "$ (day)": day.groupby("station")["maintenance"].sum(),
    "reports (July)": month3.groupby("station").size(),
    "COR (July)": month3.groupby("station")["is_cor"].sum(),
    "$ (July)": month3.groupby("station")["maintenance"].sum(),
})
counts["$ share in July (%)"] = (100 * counts["$ (July)"] / counts["reports (July)"]).round(1)
counts

In [ ]:
# One example of each per airport: a corrected report, and a report with $ (not corrected).
examples = pd.concat([
    month3[month3["is_cor"]].groupby("station").head(1),
    month3[month3["maintenance"] & ~month3["is_cor"]].groupby("station").head(1),
])
examples[["station", "valid", "is_cor", "maintenance", "metar"]]

**What:** when a report is corrected, is the **first (wrong) version** still in the file? If yes, two rows would share the same airport and time.
We check July for the three airports, then the whole project period for ATL.

**Why:** two rows for the same moment would break the "last report before `T_pred`" join (which one is "the last"?).
Silver must keep **one** row per airport and time.

In [ ]:
same_time = month3.duplicated(["station", "valid"], keep=False)   # keep=False: mark BOTH rows
print("July, 3 airports: rows sharing airport + time with another row:", int(same_time.sum()))
print("  of which corrected (COR) reports:", int((same_time & month3["is_cor"]).sum()))

# The same check on the whole ATL file (project period from config.yaml).
atl_all = read_metar("ATL", cfg["period"]["start"], cfg["period"]["end"])
twice = atl_all[atl_all.duplicated("valid", keep=False)]
print(f"ATL, whole period: {len(atl_all):,} reports | rows sharing their time: {len(twice)}")
twice[["valid", "metar"]]

**Question:** Why must we keep the special reports (`SPECI`) instead of using only the hourly ones? What should Silver do with a `COR`, and with the pairs of rows found just above (look at the two texts: which one would you keep)? Is a `$` a reason to delete a report?

**My answer:**

## A2. Decode METAR reports

**What:** read one report group by group, by hand. Then let the `metar` library (python-metar) decode five ATL reports of 2025-07-15.
**Why:** you must be able to read the raw text yourself before trusting a parser (Golden rule 10).

The storm report of ATL at 19:52 UTC, split in groups:

`KATL 151952Z 02004KT 10SM TS SCT060CB SCT100 BKN200 35/19 A3003 RMK AO2 TSB50 SLP157 OCNL LTGICCG VC NE TS VC NE MOV SW CB DSNT NE E SE MOV SW TCU N NE-SE-SW SHRA VC NE T03500189`

| Group | Text | Meaning |
|---|---|---|
| Station | `KATL` | ICAO code of Atlanta. (Not always `K` + IATA: always take it from `airportsdata`.) |
| Time | `151952Z` | Day **15**, **19:52**, `Z` = UTC. The month and the year are **not** in the report: they come from the `valid` column. |
| Wind | `02004KT` | From 020° (north-north-east) at 4 knots. With gusts it looks like `29014G27KT` = 14 kt, gusts 27 kt. `VRB` = variable direction, `00000KT` = calm. |
| Visibility | `10SM` | 10 statute miles (the largest value US stations report). `1/2SM` = half a mile. |
| Weather | `TS` | Thunderstorm, without rain at the station. `-TSRA` = thunderstorm with light rain; `+` = heavy; `VC` = in the vicinity. **No weather group = nothing to report.** |
| Clouds | `SCT060CB SCT100 BKN200` | Scattered at 6,000 ft, of type `CB` (cumulonimbus = storm cloud); scattered at 10,000 ft; broken at 20,000 ft. Height is in hundreds of feet. `FEW` < `SCT` < `BKN` < `OVC` (overcast); `CLR` = clear. |
| Temp / dew point | `35/19` | 35 °C, dew point 19 °C. `M02` = minus 2. |
| Altimeter | `A3003` | Pressure 30.03 inches of mercury. |
| Remarks | `RMK AO2 TSB50 SLP157 …` | Free text. `AO2` = automatic station with a precipitation sensor. **`TSB50` = thunderstorm began at minute 50.** `SLP157` = sea-level pressure 1015.7 hPa. `OCNL LTGICCG VC NE` = occasional lightning (in cloud, cloud to ground) in the vicinity, north-east. `TS VC NE MOV SW` = thunderstorm nearby to the north-east, moving south-west. `CB DSNT …` = storm clouds in the distance. `T03500189` = exact temperature 35.0 °C / dew point 18.9 °C. |

The **ceiling** is the height of the lowest `BKN` or `OVC` layer: here 20,000 ft (the `SCT` layers do not count).

**What:** `decode(raw, when)` calls the library on one report. Below: the storm report decoded by the library, to compare with the table above.
**Why:** the library needs the month and year from outside (`when`), because the report only holds the day and the time.

In [ ]:
def decode(raw: str, when: pd.Timestamp | None = None) -> Metar.Metar:
    """Parse one raw report with the python-metar library.

    A report only holds the day and the time, so the month and year come from `when`
    (the `valid` column). Without `when` we give January 2024: a 31-day month accepts any
    day number; the DATE of the result is then wrong and must not be used.
    The library raises an error when it cannot read a group ("strict" mode, its default).
    """
    month, year = (when.month, when.year) if when is not None else (1, 2024)
    return Metar.Metar(raw, month=month, year=year)


atl_day = day[day["station"] == "ATL"]
FIVE = ["00:52", "15:52", "19:52", "20:46", "20:52"]   # calm, corrected, storm, special, after
five = atl_day[atl_day["valid"].dt.strftime("%H:%M").isin(FIVE)]

storm_report = five.iloc[2]
print(storm_report["metar"], "\n")
print(decode(storm_report["metar"], storm_report["valid"]).string())

**What:** the five reports side by side: one column per report, one line per group, as the library reads them.
**Why:** follow the afternoon: calm → clouds building (`TCU`, `CB`) → thunderstorm → end of the storm.

In [ ]:
def decoded_fields(raw: str, when: pd.Timestamp) -> dict:
    """The main groups of one report, in words, as the library reads them."""
    obs = decode(raw, when)
    return {
        "time (UTC)": obs.time.strftime("%Y-%m-%d %H:%M"),
        "corrected": obs.mod == "COR",
        "wind": obs.wind(),
        "visibility": obs.visibility(),
        "weather": obs.present_weather() or "(no weather group)",
        "clouds": obs.sky_conditions(", "),
        "temperature": obs.temp.string("C"),
        "dew point": obs.dewpt.string("C"),
        "pressure": obs.press.string("in"),
        "remarks": obs.remarks(" | "),
    }


side_by_side = pd.DataFrame({
    f"{when:%H:%M}": decoded_fields(raw, when)
    for raw, when in zip(five["metar"], five["valid"], strict=True)   # walk both columns together
})
side_by_side

**Your turn.** The cell below prints one more ATL report (2025-07-01, 19:00 UTC). Decode it group by group in your answer, **then** remove the `#` to compare with the library.

In [ ]:
to_decode = read_metar("ATL", "2025-07-01 19:00", "2025-07-01 19:01").iloc[0]
print(to_decode["metar"])
# When your answer is written, remove the # below to compare with the library:
# print(decode(to_decode["metar"], to_decode["valid"]).string())

**Question:** Decode the report printed above, group by group (station, time, wind, visibility, weather, clouds, temperature / dew point, altimeter, and what you understand of the remarks). What is the ceiling? Is it an hourly report or a special one?

**My answer:**

## A3. From text to numbers: `parse_metar_features`

**What:** one function that turns a raw report into a small dictionary:

| Key | Meaning |
|---|---|
| `wind_kt` | wind speed in knots |
| `gust_kt` | gust speed in knots; **NaN when no gust is reported** (most of the time) |
| `vis_sm` | visibility in statute miles |
| `ceiling_ft` | height of the lowest `BKN` / `OVC` / `VV` layer in feet; **NaN when there is no ceiling** |
| `ts_in_weather` | `TS` in a weather group (`TS`, `-TSRA`, `VCTS`…) = a thunderstorm **at or next to** the airport now |
| `ts_in_remarks` | a thunderstorm **hint** in the remarks: `TS`, `CB`, `LTG…`, `TSB..` / `TSE..` |
| `thunderstorm` | `ts_in_weather` **or** `ts_in_remarks` (the wide flag) |
| `precip` | precipitation in a weather group (`RA` rain, `DZ` drizzle, `SN` snow, `GR` hail, `PL` ice pellets…) |
| `parse_ok` | `False` when the report could not be read |

**Why:** these are the "parsed weather numbers" that XGBoost will use (Phase 2), and the reference for the text encoder (Phase 3).
The function must **never crash**: on any error it returns NaN everywhere and `parse_ok = False`, so we can **count** the failures instead of losing rows silently (Golden rule 5).

In [ ]:
NAN = float("nan")
FEATURES = ["wind_kt", "gust_kt", "vis_sm", "ceiling_ft",
            "ts_in_weather", "ts_in_remarks", "thunderstorm", "precip"]
# Thunderstorm hints in the remarks, as separate words (\b = word border):
# TS, CB (storm cloud), LTG... (lightning: LTGICCG, LTGCG...), TSB12 / TSE44 (thunderstorm
# began / ended at minute 12 / 44; also chained forms like TSB1154E09).
TS_IN_REMARKS = re.compile(r"\b(TS|CB|LTG\w*|TS[BE]\d[\dBE]*)\b")


def parse_metar_features(raw: str) -> dict:
    """Raw METAR -> numbers and flags. Never raises: an unreadable report gives NaN
    everywhere and parse_ok = False, so that failures can be counted."""
    try:
        obs = decode(raw)
        if obs.station_id is None:                 # e.g. an empty text
            raise ValueError("no station found")
        # Cloud layers are (cover, height, cloud type). The ceiling is the lowest layer that
        # covers more than half of the sky: BKN, OVC, or VV (sky hidden, vertical visibility).
        layers = [height.value("FT") for cover, height, _type in obs.sky
                  if cover in ("BKN", "OVC", "VV") and height is not None]
        # Weather groups are (intensity, description, precipitation, obscuration, other):
        # "-TSRA" -> ("-", "TS", "RA", None, None); "VCTS" -> ("VC", "TS", "", None, None).
        ts_in_weather = any("TS" in (description or "")
                            for _i, description, _p, _o, _x in obs.weather)
        precip = any(bool(precipitation) for _i, _d, precipitation, _o, _x in obs.weather)
        # Everything after " RMK " is the remarks part (free text).
        _body, _sep, remarks = raw.partition(" RMK ")
        ts_in_remarks = TS_IN_REMARKS.search(remarks) is not None
        return {
            "wind_kt": obs.wind_speed.value("KT") if obs.wind_speed is not None else NAN,
            "gust_kt": obs.wind_gust.value("KT") if obs.wind_gust is not None else NAN,
            "vis_sm": obs.vis.value("SM") if obs.vis is not None else NAN,
            "ceiling_ft": min(layers) if layers else NAN,
            "ts_in_weather": ts_in_weather,
            "ts_in_remarks": ts_in_remarks,
            "thunderstorm": ts_in_weather or ts_in_remarks,
            "precip": precip,
            "parse_ok": True,
        }
    except Exception:       # any error at all: the notebook goes on, the failure is counted
        return {**dict.fromkeys(FEATURES, NAN), "parse_ok": False}

**What:** try the function on four texts: a calm report, the storm report, a **real** ATL report with a typo in the wind group (`080/05KT`, from another month), and a text that is not a METAR.
**Why:** check by hand that the numbers match the raw text, and that bad input gives NaN + `parse_ok = False` instead of an error.

In [ ]:
tests = {
    "calm": five.iloc[0]["metar"],
    "storm": storm_report["metar"],
    "typo in the wind group": ("KATL 200937Z COR 080/05KT 1/2SM +TSRA FG BKN006 OVC016CB 22/22 "
                               "A3013 RMK AO2 RAB06 TSB29 PRESRR OCNL LTGIC OHD-W TS OHD-W"),
    "not a METAR": "hello",
}
for name, raw in tests.items():
    print(f"{name:>22}: {raw}")
pd.DataFrame({name: parse_metar_features(raw) for name, raw in tests.items()}).T

**What:** apply the function to every ATL report of July 2025 and count the failures.
**Why:** the failure share goes into the data-quality report. A high share would mean the parser is not good enough for our data.

In [ ]:
def with_features(reports: pd.DataFrame) -> pd.DataFrame:
    """The same reports with one more column per feature (same rows, same order)."""
    features = pd.DataFrame([parse_metar_features(raw) for raw in reports["metar"]],
                            index=reports.index)
    return pd.concat([reports, features], axis=1)      # axis=1: put the columns side by side


atl = with_features(read_metar("ATL", MONTH_START, MONTH_END))
n_fail = int((~atl["parse_ok"]).sum())
print(f"METAR rows read (ATL, July 2025): {len(atl):,}")
print(f"parse failures: {n_fail} ({100 * n_fail / len(atl):.2f} %)")
atl[["valid", "metar", *FEATURES]].head(3)

**What:** `describe()` of the four numbers, and the share of reports with each flag.
**Why:** look at `count`: it is the number of reports **with a value**. For `gust_kt` and `ceiling_ft` it is much lower than the number of reports:
there NaN means "no gust" / "no ceiling", which is information, not a hole in the data.

In [ ]:
NUMBERS = ["wind_kt", "gust_kt", "vis_sm", "ceiling_ft"]
FLAGS = ["ts_in_weather", "ts_in_remarks", "thunderstorm", "precip"]

print("share of ATL reports with each flag (%):")
# .eq(True) is True only for a real True (a failed report, with NaN, counts as False)
print((100 * atl[FLAGS].eq(True).mean()).round(1).to_string())
atl[NUMBERS].describe().round(1)

**What:** wind (line) and gusts (dots) over the month at ATL. A red bar marks every report with `TS` in a weather group.
**Why:** see whether gusts and thunderstorms come together, and how rare stormy hours are in a month.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ts_times = atl.loc[atl["ts_in_weather"].eq(True), "valid"]
# One vertical bar per thunderstorm report, over the full height of the chart.
ax.vlines(ts_times, 0, 1, transform=ax.get_xaxis_transform(), color="red", alpha=0.25,
          linewidth=2, label="report with TS (weather group)")
ax.plot(atl["valid"], atl["wind_kt"], color="steelblue", linewidth=1, label="wind (kt)")
ax.scatter(atl["valid"], atl["gust_kt"], color="darkorange", s=14, zorder=3, label="gust (kt)")
ax.set_xlabel("July 2025 (UTC)")
ax.set_ylabel("knots")
ax.set_title("ATL: wind, gusts and thunderstorm reports")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

**Question:** `ts_in_remarks` is true much more often than `ts_in_weather`. Why? (Read a few remarks: `CB DSNT`, `TS DSIPTD`, `TSE44`.) Which of the two flags means "a thunderstorm is at the airport now"? And why is NaN the right value for `gust_kt` when there is no gust, instead of 0?

**My answer:**

## A4. Thunderstorm days at the 10 hubs

**What:** parse July 2025 for the 10 hubs, then count the **storm hours** per hub and per day:
a storm hour = one UTC hour with at least one report that has `TS` in a weather group.
**Why:** storms are the rare events the radar and text modalities exist for. We need to know how many there are, and we need
one clear **storm day** and one **calm day** at ATL to test the radar in Part B.

Days are **UTC days**: an evening storm in Atlanta (after 20:00 local time) falls on the next UTC day.

In [ ]:
hubs_july = pd.concat(
    [with_features(read_metar(hub, MONTH_START, MONTH_END)).assign(hub=hub) for hub in HUBS],
    ignore_index=True)
failed = hubs_july[~hubs_july["parse_ok"]]
print(f"METAR rows read (10 hubs, July 2025): {len(hubs_july):,}")
print(f"parse failures: {len(failed)} ({100 * len(failed) / len(hubs_july):.2f} %)")
failed[["hub", "valid", "metar"]]      # read them: what is wrong in each?

In [ ]:
ts = hubs_july[hubs_july["ts_in_weather"].eq(True)].copy()
ts["day"] = ts["valid"].dt.strftime("%m-%d")
ts["hour"] = ts["valid"].dt.floor("h")         # 19:52 -> 19:00
all_days = pd.date_range(MONTH_START, MONTH_END, inclusive="left").strftime("%m-%d")

# nunique = number of DIFFERENT hours: three TS reports in the same hour count once.
# reindex adds the hubs and days that have no storm at all (filled with 0).
storm_hours = (ts.groupby(["hub", "day"])["hour"].nunique().unstack(fill_value=0)
               .reindex(index=HUBS, columns=all_days, fill_value=0))

print("storm hours per hub, July 2025:")
print(storm_hours.sum(axis=1).sort_values(ascending=False).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
im = ax.imshow(storm_hours.to_numpy(), cmap="Reds", aspect="auto")
ax.set_xticks(range(len(all_days)), [d[3:] for d in all_days])      # "07-15" -> "15"
ax.set_yticks(range(len(HUBS)), HUBS)
# Write the number in every cell that is not 0.
for (i, j), n in np.ndenumerate(storm_hours.to_numpy()):
    if n > 0:
        ax.text(j, i, n, ha="center", va="center", fontsize=8,
                color="white" if n >= 4 else "black")
ax.set_xlabel("Day of July 2025 (UTC)")
ax.set_title("Storm hours per hub and per day (hours with a TS report)")
fig.colorbar(im, ax=ax, label="storm hours")
plt.tight_layout()
plt.show()

**What:** one line per day at ATL, then two days chosen **by code** (not by hand):
- `STORM_DAY`: the day with the most storm hours (if equal: the most TS reports);
- `CALM_DAY`: a day with no thunderstorm hint at all (not even in the remarks), no precipitation, and the lowest maximum wind.

**Why:** Part B compares the radar of these two days. Writing the choice as code keeps it honest and repeatable.

In [ ]:
assert atl["parse_ok"].all(), "a failed report would need to be removed before counting"
atl["day"] = atl["valid"].dt.strftime("%Y-%m-%d")
# The hour of the report, kept only for TS reports (empty otherwise): used to count storm hours.
atl["ts_hour"] = atl["valid"].dt.floor("h").where(atl["ts_in_weather"])

atl_days = atl.groupby("day").agg(
    reports=("metar", "size"),
    ts_hours=("ts_hour", "nunique"),            # different hours with a TS report
    ts_reports=("ts_in_weather", "sum"),        # sum of True/False = number of True
    any_ts_hint=("thunderstorm", "sum"),        # wide flag: weather group OR remarks
    precip_reports=("precip", "sum"),
    max_wind_kt=("wind_kt", "max"),
    max_gust_kt=("gust_kt", "max"),
    min_vis_sm=("vis_sm", "min"),
)
atl_days

In [ ]:
# kind="stable": when two days are equal, the earlier one stays first (same result at every run).
by_storm = atl_days.sort_values(["ts_hours", "ts_reports"], ascending=False, kind="stable")
STORM_DAY = by_storm.index[0]

quiet = atl_days[(atl_days["any_ts_hint"] == 0) & (atl_days["precip_reports"] == 0)]
CALM_DAY = quiet.sort_values("max_wind_kt", kind="stable").index[0]

print("STORM_DAY =", STORM_DAY, "| CALM_DAY =", CALM_DAY, "(UTC days, ATL)")
atl_days.loc[[STORM_DAY, CALM_DAY]]

**Question:** Which hub had the most storm hours in July 2025, and which had none? Does it match what you know (or can look up) about US summer weather: afternoon thunderstorms in the South-East, the dry West coast, the monsoon in the South-West? What does it mean for a model trained on only one month?

**My answer:**

## A5. TAF: the forecast

**What:** the KATL TAF file for July 2025. A TAF is a **forecast** for one airport for the next 24–30 hours, written by a forecaster.
In this file there is **one row per forecast group, not one row per TAF**: a TAF is cut into pieces, each valid for a part of the period.

| Column | Meaning |
|---|---|
| `valid` | **issue time** of the TAF (UTC): the moment it became known |
| `fx_valid` | start of the period this group forecasts |
| `fx_valid_end` | end of the period, only for `TEMPO` / `PROB` groups |
| `raw` | the text of the group |
| `ftype` | kind of group (see below) |
| `product_id` | identifier of the TAF: **all rows of one TAF share it** |
| `is_amendment` | True if this TAF replaces an earlier one |

**Why:** for a prediction made 2 h before departure, the weather **at departure** is not known yet. The TAF is the only weather text that speaks about the future.
We must understand its structure to pick "the latest TAF issued at or before `T_pred`".

In [ ]:
taf = read_taf("KATL", MONTH_START, MONTH_END)
print(f"rows (forecast groups): {len(taf):,} | TAFs (product_id): {taf['product_id'].nunique()}"
      f" | type of is_amendment: {taf['is_amendment'].dtype}")
TAF_COLS = ["valid", "fx_valid", "fx_valid_end", "ftype", "raw", "is_amendment", "product_id"]
taf[TAF_COLS].head(8)

**What:** the kinds of group (`ftype`), and a search for `BECMG` and `PROB40` in the text.
**Why:** to know which kinds really appear in our data before writing code for them.

- `Observation` (IEM's name) is the **first group** of the TAF: the conditions from the issue time on. It is still a forecast, not an observation.
- `Forecast` = a `FM` group.
- `Temporary` = a `TEMPO` group. In part of the file the word `TEMPO` is missing from `raw`; the `ftype` / `is_tempo` columns still say it.
- `Probability 30` = a `PROB30` group.

In [ ]:
print(taf["ftype"].value_counts().to_string())
print("\nrows whose text contains BECMG :", int(taf["raw"].str.contains("BECMG").sum()))
print("rows whose text contains PROB40:", int(taf["raw"].str.contains("PROB40").sum()))

### TAF vocabulary, in simple words

| Word | Meaning | Example |
|---|---|---|
| `FM` (from) | **From** this time on, the weather changes to this (a clear, lasting change). Valid until the next `FM`. | `FM011900 23008KT P6SM -SHRA BKN040` = from day 01 at 19:00 UTC: wind 230° 8 kt, visibility more than 6 miles, light rain showers, broken clouds at 4,000 ft |
| `TEMPO` | **Temporarily**, for short moments inside this period (each under an hour), the weather may become this. The base forecast stays valid in between. | `TEMPO 0121/0201 VRB15G30KT 4SM TSRA OVC025CB` = between day 01 21:00 and day 02 01:00, at times: gusts 30 kt, thunderstorm with rain |
| `PROB30` / `PROB40` | A **30 % / 40 % chance** of this weather in this period. | `PROB30 0118/0122 4SM -TSRA BR BKN040CB` |
| `BECMG` (becoming) | The weather changes **gradually** during this period. Common in European TAFs; US TAFs use `FM` instead (see the count above). | `BECMG 0110/0112 27015KT` |
| `AMD` (amended) | This TAF **replaces** the previous one, because the forecast changed. In `product_id` an amendment ends with `-AAA`, `-AAB`, `-AAC`… (1st, 2nd, 3rd amendment of the same routine TAF); `-CCA` = a correction (typing error), `-RRA` = sent late. | |
| `P6SM` | Visibility **more than** 6 miles (TAFs do not go higher). | |

**What:** one line per **TAF** (grouped by `product_id`): issue time, number of groups, and its kind read from the end of the `product_id`
(nothing = routine, `-AA.` = amendment, `-CC.` = correction, `-RR.` = sent late). Then the kind compared with the `is_amendment` column.
**Why:** the `is_amendment` column should say the same thing as the `product_id`. If it is sometimes **empty**, we cannot rely on it alone.

In [ ]:
def taf_kind(product_id: str) -> str:
    """routine / amendment / correction / delayed, read from the end of the product_id."""
    parts = product_id.split("-")      # [..., 'TAFATL'] or [..., 'TAFATL', 'AAA']
    suffix = parts[4] if len(parts) == 5 else ""
    kinds = {"": "routine", "AA": "amendment", "CC": "correction", "RR": "delayed"}
    return kinds.get(suffix[:2], "other")


one_issue_time = taf.groupby("product_id")["valid"].nunique().max() == 1
print("every product_id has exactly one issue time:", one_issue_time)

tafs = taf.groupby("product_id").agg(
    issued=("valid", "first"), groups=("raw", "size"), is_amendment=("is_amendment", "first"))
tafs["kind"] = [taf_kind(pid) for pid in tafs.index]
tafs = tafs.sort_values("issued")

# Lines: kind from the product_id. Columns: the is_amendment column ("empty" = no value).
pd.crosstab(tafs["kind"], tafs["is_amendment"].astype("string").fillna("empty"))

**What:** number of TAFs per day, by kind, and the hour of the day (UTC) at which they are issued.
**Why:** the textbook says "4 TAFs per day" (for 00, 06, 12 and 18 UTC). If KATL issues many more, the choice "which TAF did we know at `T_pred`?" matters a lot.

In [ ]:
per_day = (tafs.groupby([tafs["issued"].dt.strftime("%m-%d"), "kind"]).size()
           .unstack(fill_value=0))
per_day["all kinds"] = per_day.sum(axis=1)
print("TAFs per day at KATL, July 2025:")
print(per_day.describe().loc[["mean", "min", "max"]].round(1).to_string())

by_hour = (tafs.groupby([tafs["issued"].dt.hour, "kind"]).size().unstack(fill_value=0)
           .reindex(range(24), fill_value=0))
ax = by_hour.plot.bar(stacked=True, figsize=(10, 3.5), rot=0, width=0.8)
ax.set_xlabel("Issue hour (UTC)")
ax.set_ylabel("TAFs in July 2025")
ax.set_title("KATL: when are TAFs issued?")
ax.legend(title="kind", loc="upper left", bbox_to_anchor=(1.0, 1.0))   # outside, on the right
plt.tight_layout()
plt.show()

**What:** rebuild **one full TAF** from its rows: the routine TAF of `STORM_DAY` with the most groups.
`taf_groups_in_order` sorts the groups by start time and gives each one the period it covers:
- a **base** group (the first group, or `FM`) lasts **until the next base group starts**; the last one lasts until the end of the TAF (not in the file: shown as `NaT`);
- a `TEMPO` / `PROB30` group has its own start and end, and sits **on top of** the base group.

**Why:** rows of one TAF are not always stored in time order. And to answer "what does this TAF say for 19:00?", we need the period of every group.

In [ ]:
def taf_groups_in_order(groups: pd.DataFrame) -> pd.DataFrame:
    """The groups of ONE TAF in time order, with the period (from, until) each one covers."""
    g = groups.copy()
    g["is_base"] = g["ftype"].isin(["Observation", "Forecast"])     # first group or FM
    # By start time; at the same start time the base group first (True before False).
    g = g.sort_values(["fx_valid", "is_base"], ascending=[True, False])
    # A base group ends when the next base group starts. shift(-1) = "the value of the row below".
    base = g[g["is_base"]]
    g.loc[base.index, "fx_valid_end"] = base["fx_valid"].shift(-1)
    g = g.rename(columns={"fx_valid": "from", "fx_valid_end": "until"})
    return g[["from", "until", "ftype", "raw"]].reset_index(drop=True)


routine_that_day = tafs[(tafs["kind"] == "routine")
                        & (tafs["issued"].dt.strftime("%Y-%m-%d") == STORM_DAY)]
one_id = routine_that_day["groups"].idxmax()        # the product_id with the most groups
print("TAF:", one_id, "| issued:", tafs.loc[one_id, "issued"], "|", tafs.loc[one_id, "kind"])
taf_groups_in_order(taf[taf["product_id"] == one_id])

**What:** a check of IEM's **parsed** columns (`presentwx`, `skyc`…) on the first group of each TAF: does `presentwx` contain a thunderstorm (`TS`) that is **not** in the group's own `raw` text?
**Why:** if the parsed columns of one group contain weather that belongs to **another** group, we cannot use them blindly. The `raw` text is the reference.

In [ ]:
first_groups = taf[taf["ftype"] == "Observation"]
odd = first_groups[first_groups["presentwx"].str.contains("TS")
                   & ~first_groups["raw"].str.contains("TS")]
print(f"first groups: {len(first_groups)} "
      f"| 'TS' in presentwx but not in their own raw text: {len(odd)}")
odd[["valid", "raw", "presentwx", "skyc"]].head(3)

**Question:** How many TAFs does KATL issue per day, and how many of them are routine? At which UTC hours do the routine ones come, and what do you notice about the hours of the amendments? Why can we not simply take "the TAF of the day" for a flight? Which column would you trust to recognise an amendment, and why?

**My answer:**

## A6. "The latest TAF at `T_pred`": a demo on one departure

**What:** a **made-up** departure from ATL on `STORM_DAY`, scheduled at the full hour that follows the worst thunderstorm report of the day
(the TS report with the lowest visibility). This is only to understand the rule; it is **not** pipeline code.

- `DEP_UTC` = scheduled departure (UTC).
- `T_PRED` = `DEP_UTC` − 2 h = **the prediction moment**. The model lives at `T_PRED`: it knows nothing that happened after it.

**Why:** this is Golden rule 2 on a real example. We will see what was known at `T_PRED`, and what was not.

In [ ]:
day_start = utc(STORM_DAY)
on_storm_day = atl[(atl["valid"] >= day_start) & (atl["valid"] < day_start + pd.Timedelta(days=1))]
worst = on_storm_day[on_storm_day["ts_in_weather"]].sort_values("vis_sm", kind="stable").iloc[0]

DEP_UTC = worst["valid"].ceil("h")                      # the full hour at or after that report
T_PRED = DEP_UTC - pd.Timedelta(hours=T_PRED_H)         # the prediction moment
dep_local = DEP_UTC.tz_convert(AIRPORTS["ATL"]["tz"])   # the same moment on Atlanta's clock

print("worst TS report  :", worst["valid"], "|", worst["metar"][:60], "...")
print("departure (UTC)  :", DEP_UTC, "=", dep_local.strftime("%H:%M"), "local time in Atlanta")
print("T_pred    (UTC)  :", T_PRED)

**What:** among all TAFs **issued at or before `T_PRED`**, take the latest one. Then, inside it, find the group(s) that cover `DEP_UTC`:
the base group in force at that time, plus any `TEMPO` / `PROB30` group whose period contains it.
The `assert` line stops the notebook if the TAF were issued after `T_PRED`.

**Why:** two different times are used, and they must not be confused:
- the **issue time** (`valid`) must be **≤ `T_PRED`**: we can only use a forecast that already existed;
- the **forecast period** (`from` / `until`) must contain the **departure time**: that is the moment we want to know about.

In [ ]:
# TAFs issued in the 2 days before T_pred (far more than needed) up to 1 day after departure.
taf_near = read_taf("KATL", T_PRED - pd.Timedelta(days=2), DEP_UTC + pd.Timedelta(days=1))

known = taf_near[taf_near["valid"] <= T_PRED]            # issued at or before T_pred
# The latest issue time; if two TAFs share it (a correction), the product_id decides.
latest_id = known.sort_values(["valid", "product_id"])["product_id"].iloc[-1]
latest_taf = known[known["product_id"] == latest_id]
issue_time = latest_taf["valid"].iloc[0]
assert issue_time <= T_PRED, "this TAF was issued after T_pred: it would be a leak"

next_issue = taf_near.loc[taf_near["valid"] > T_PRED, "valid"].min()
print("latest TAF known at T_pred:", latest_id, f"({taf_kind(latest_id)})")
print("  issued:", issue_time, "| age at T_pred:", T_PRED - issue_time)
print("next TAF (NOT allowed, issued after T_pred):", next_issue)

groups = taf_groups_in_order(latest_taf)
# A group covers the departure if it starts at or before it and has not ended yet
# (an empty "until" = the last base group, valid until the end of the TAF).
covers_departure = (groups["from"] <= DEP_UTC) & (groups["until"].isna()
                                                  | (groups["until"] > DEP_UTC))
groups.assign(covers_departure=covers_departure)

**What:** two METAR reports side by side:
- the **last report at or before `T_PRED`**: what the model is allowed to see;
- the **report at departure time**: what really happened. The model must **never** see it.

**Why:** compare them with the TAF above. Which text warned about the weather at departure?

In [ ]:
def last_report_at_or_before(reports: pd.DataFrame, when: pd.Timestamp):
    """The most recent report with valid <= when (reports sorted by time); None if none."""
    earlier = reports[reports["valid"] <= when]
    return earlier.iloc[-1] if len(earlier) else None


metar_known = last_report_at_or_before(atl, T_PRED)
metar_at_dep = last_report_at_or_before(atl, DEP_UTC)
assert metar_known["valid"] <= T_PRED

compare = pd.DataFrame({
    "last METAR <= T_pred (allowed)": metar_known,
    "METAR at departure (FORBIDDEN as input)": metar_at_dep,
})
compare.loc[["valid", "metar", "wind_kt", "gust_kt", "vis_sm", "ceiling_ft",
             "ts_in_weather", "thunderstorm", "precip"]]

**Why the first METAR is allowed and the second is not**

- At `T_PRED` the operations controller (and so the model) can read every report **already sent**: the last METAR at or before `T_PRED` is one of them.
- The METAR of the departure time **does not exist yet** at `T_PRED`. Using it would be **leakage**: the model would look excellent in tests,
  and be useless in real life, where that report arrives two hours too late.
- The METAR says what the weather **is**; only the TAF says what it **will be**. So the model gets: the last METAR ≤ `T_PRED` (the present),
  **and** the latest TAF issued ≤ `T_PRED`, read at the departure time (the expected future).
- The observed weather at departure is still useful **after** the flight: to evaluate (storm-day MAE) and for the replay demo. Never as an input.

**Question:** On this example: what does the last METAR before `T_PRED` say, what does the TAF say for the departure time, and what really happened at departure? Which of the two texts gave the better warning? What would be wrong with a model trained with the departure-time METAR as an input?

**My answer:**

## A7. Summary of Part A: what Silver must do

**METAR**
- **All times in UTC**; one row per airport and time. Where two rows share the same airport and time, keep one: the corrected report (`COR`) if there is one,
  otherwise the complete one (not the short `AUTO` copy). In these IEM files the first version of a corrected report is usually already gone: keep a `is_corrected` flag.
- **Keep the specials (`SPECI`)**: they are the reports sent when the weather changes fast, exactly the moments we care about. They are recognised by their minute, the word is not in the file.
- **Flag `$`** (station needs maintenance) in a column; do not delete the report.
- **Thunderstorm:** read the weather groups **and** the remarks. Keep two flags: `ts_in_weather` (at the airport now) and `ts_in_remarks` (a hint: distant, just ended, lightning).
- **Parser failures** are counted and reported, never dropped silently. NaN for "no gust" / "no ceiling" is information, not a missing value.
- **Join rule:** last report with `valid` ≤ `T_pred`. Never a later one.

**TAF**
- One row per forecast **group**; the TAF is the `product_id`. Rows of one TAF are not always in time order: sort them.
- **Join rule:** the latest TAF with issue time (`valid`) ≤ `T_pred`; inside it, the base group in force at the **target time** (scheduled departure for the origin, scheduled arrival for the destination), plus the `TEMPO` / `PROB30` groups whose period contains it.
- Recognise amendments from the **`product_id`** (`-AAA`, `-AAB`…), because `is_amendment` is sometimes empty. If two TAFs share an issue time, take the correction.
- Use the **`raw`** text as the reference; IEM's parsed columns of the first group can contain weather of another group (to check again in Silver).
- Store the **age** of the TAF at `T_pred`; a very old "latest TAF" means missing data, not a forecast.
- **Missing TAF:** 8 airports have none at all (BLV, BQK, GRK, HHH, MVY, SHD, VPS, YUM). The model needs a `has_taf` flag, like `has_radar` in Part B. Palm Beach has two raw files (`KPBI`, `KDJT`) to merge.

---
# PART B — Radar

A weather radar sends a beam and measures how much comes back from rain drops, hail or snow. IEM glues all US radars into one **national image** every 5 minutes.
We downloaded one image per hour, cut a **384 × 384 pixel crop** around each airport, and kept only the crops (the national images are too big to keep).

## B1. One national image and its `.wld` file

**What:** list the national images kept for debugging (one day only: `radar.keep_national_days` in `config.yaml`), then read the `.wld` file.
**Why:** the PNG alone is only a grid of pixels. The `.wld` ("world file") says **where on Earth** each pixel is. Without it we cannot find Atlanta in the image.

In [ ]:
NAT_DAY = pd.Timestamp(cfg["radar"]["keep_national_days"][0])       # the debug day in config.yaml
NAT_DIR = RAW / "radar_national" / f"{NAT_DAY:%Y}" / f"{NAT_DAY:%m}" / f"{NAT_DAY:%d}"

pngs = sorted(NAT_DIR.glob(f"{PRODUCT}_*.png"))
wlds = sorted(NAT_DIR.glob(f"{PRODUCT}_*.wld"))
# File name: n0q_YYYYMMDDHHMM.png -> the 4 characters before the last 2 are "HH" + "MM"
print("folder:", NAT_DIR)
print(f"{len(pngs)} PNG, UTC hours:", " ".join(p.stem[-4:-2] for p in pngs))
print(f"{len(wlds)} .wld:", [w.name for w in wlds])
print(f"size of one PNG: {pngs[0].stat().st_size / 1e6:.1f} MB")

**What:** the six numbers of the `.wld` file, compared with `radar.expected_wld` in `config.yaml`.
**Why:** `download_radar` refuses to run if these numbers change: every crop would be cut at the wrong place, without any error message.

Only one `.wld` was kept for the day (the script reads the first one of the day): the grid is the same for every hour.

A `.wld` file has 6 lines, always in this order:

| Line | Name | Value here | Meaning |
|---|---|---|---|
| 1 | A | 0.005 | width of one pixel, in **degrees of longitude** (going right = going east) |
| 2 | D | 0 | rotation term: 0 = the image is not rotated |
| 3 | B | 0 | rotation term: 0 = the image is not rotated |
| 4 | E | −0.005 | height of one pixel, in **degrees of latitude**. **Negative**: going down in the image = going south |
| 5 | C | −126.0 | longitude of the centre of the **upper-left** pixel |
| 6 | F | 50.0 | latitude of the centre of the **upper-left** pixel |

So the image starts in the north-west corner (126° W, 50° N, in the Pacific off Vancouver Island) and each pixel is 0.005° wide and high.
The coordinates are plain longitude / latitude (EPSG:4326).

In [ ]:
wld = [float(x) for x in wlds[0].read_text().split()]
names = ["A  pixel width (deg lon)", "D  rotation", "B  rotation",
         "E  pixel height (deg lat)", "C  upper-left longitude", "F  upper-left latitude"]
print("matches radar.expected_wld:", np.allclose(wld, EXPECTED_WLD))   # allclose: equal numbers
pd.DataFrame({"this .wld": wld, "expected (config.yaml)": EXPECTED_WLD}, index=names)

**What:** open one national PNG (20:00 UTC: section A1 showed a thunderstorm report at ATL at 19:52 that day) and check its size against `radar.expected_shape`.
**Why:** the `.wld` only gives the corner and the pixel size. If the image had another size, the grid would no longer describe it.

The PNG is a **palette image** (mode `P`): each pixel holds a **number from 0 to 255**, not a colour. A table inside the file (the palette) says which colour to show for each number.
**We keep the numbers** (they carry the radar intensity, section B3) and use the palette only to draw pictures with IEM's own colours.

In [ ]:
def read_national(path: Path) -> tuple[np.ndarray, np.ndarray]:
    """National PNG -> (2-D array of pixel values 0-255, palette = 256 RGB colours)."""
    with Image.open(path) as img:
        assert img.mode == "P", f"unexpected image mode {img.mode}"      # P = palette image
        palette = np.array(img.getpalette(), dtype=np.uint8).reshape(-1, 3)
        return np.asarray(img), palette


NAT_TIME = utc(f"{NAT_DAY:%Y-%m-%d} 20:00")
national, palette = read_national(NAT_DIR / f"{PRODUCT}_{NAT_TIME:%Y%m%d%H%M}.png")
RADAR_CMAP = ListedColormap(palette / 255)       # value -> the colour IEM gave it

n_rows, n_cols = national.shape
print("image shape (rows, columns):", national.shape, "| expected:", EXPECTED_SHAPE,
      "| match:", national.shape == EXPECTED_SHAPE)
print("type:", national.dtype, "| smallest value:", national.min(),
      "| largest value:", national.max())
print(f"pixels with value 0: {100 * (national == 0).mean():.1f} %")

# The four borders of the image, from the .wld numbers.
lon_left, lat_top = wld[4], wld[5]
lon_right = lon_left + n_cols * wld[0]
lat_bottom = lat_top + n_rows * wld[3]
print(f"covers longitude {lon_left} to {lon_right}, latitude {lat_bottom} to {lat_top}")

**What:** the whole US image, made 10 times smaller in each direction, with the 10 hubs marked.
**Why:** the full image has 5,400 × 12,200 pixels, far more than a screen. To shrink it we keep the **strongest** value of each 10 × 10 block:
simply keeping one pixel out of ten could make thin storm lines disappear.

In [ ]:
STEP = 10
# reshape cuts the image into blocks of STEP x STEP pixels; max keeps the strongest of each.
small = national.reshape(n_rows // STEP, STEP, n_cols // STEP, STEP).max(axis=(1, 3))

fig, ax = plt.subplots(figsize=(13, 6.5))
# extent puts longitude / latitude on the axes. aspect 1.25: at these latitudes one degree of
# longitude is shorter on the ground than one degree of latitude.
ax.imshow(small, cmap=RADAR_CMAP, vmin=0, vmax=255, interpolation="nearest",
          extent=(lon_left, lon_right, lat_bottom, lat_top), aspect=1.25)
for hub in HUBS:
    lat, lon = AIRPORTS[hub]["lat"], AIRPORTS[hub]["lon"]
    ax.plot(lon, lat, marker="o", markersize=5, markerfacecolor="none", color="white")
    ax.text(lon + 0.4, lat + 0.4, hub, color="white", fontsize=9)
ax.set_xlabel("Longitude (degrees)")
ax.set_ylabel("Latitude (degrees)")
ax.set_title(f"National radar image {NAT_TIME:%Y-%m-%d %H:%M} UTC (black = value 0)")
plt.tight_layout()
plt.show()

**Question:** Which of the six `.wld` numbers tell you the pixel size, and which the position of the image? Why is E negative? Looking at the map: which of our hubs had rain or storms nearby at that moment, and which US states are **not** on this image at all?

**My answer:**

## B2. From latitude / longitude to a pixel, and the crop around ATL

**What:** `latlon_to_pixel(lat, lon, wld)` gives the row and the column of the pixel that contains a point.

The `.wld` says: `lon = C + col × A` and `lat = F + row × E`. We turn the two formulas around:

- `col = (lon − C) / A` → how many pixel widths east of the left border;
- `row = (lat − F) / E` → how many pixel heights **below** the top border. Both `lat − F` and `E` are negative (we are south of 50° N), so the result is positive.

We round to the nearest whole number, because a pixel index is a whole number.

**Why:** this is the only link between an airport and the image. If it is wrong, the model looks at the weather of another place.

In [ ]:
def latlon_to_pixel(lat: float, lon: float, wld: list[float]) -> tuple[int, int]:
    """(row, column) of the pixel that contains the point (lat, lon)."""
    a, _d, _b, e, c, f = wld          # the 6 numbers, in the order of the file
    col = int(round((lon - c) / a))
    row = int(round((lat - f) / e))
    return row, col


atl_lat, atl_lon = AIRPORTS["ATL"]["lat"], AIRPORTS["ATL"]["lon"]
atl_row, atl_col = latlon_to_pixel(atl_lat, atl_lon, wld)
print(f"ATL: lat {atl_lat}, lon {atl_lon} -> row {atl_row}, column {atl_col}")
print(f"image size: {n_rows} rows x {n_cols} columns")

**What:** cut a `CROP_PX` × `CROP_PX` square around ATL from the national image **ourselves**, then compare it with the crop stored by `download_radar` for the same hour.
`np.array_equal` is True only if **every** pixel is the same.

- The crop takes `CROP_PX / 2` pixels on each side of the airport, so the airport is at pixel `(CROP_PX / 2, CROP_PX / 2)` = (192, 192) of the crop.
- If the square went beyond the border of the national image, the part outside would be filled with **255** ("outside the image"), never 0 (0 = "no rain").
- Stored crops: one `.npz` file per day; inside, one array per key `<IATA>_<YYYYMMDDHH>` (UTC hour). `np.load` opens the file and reads **only the keys we ask for**.

**Why:** the stored crops were made weeks ago by a script. Rebuilding one by hand and getting the identical array proves we understand exactly what they contain.

In [ ]:
OUTSIDE = 255     # value reserved for "outside the national image"


def cut_crop(img: np.ndarray, row: int, col: int, size: int) -> np.ndarray:
    """Square of `size` pixels centred on (row, col); pixels outside the image get 255."""
    half = size // 2
    out = np.full((size, size), OUTSIDE, dtype=np.uint8)       # start with "outside" everywhere
    r0, c0 = row - half, col - half                            # upper-left corner of the square
    # the part of the square that really is inside the image
    rr0, rr1 = max(r0, 0), min(r0 + size, img.shape[0])
    cc0, cc1 = max(c0, 0), min(c0 + size, img.shape[1])
    out[rr0 - r0:rr1 - r0, cc0 - c0:cc1 - c0] = img[rr0:rr1, cc0:cc1]
    return out


def crop_file(when) -> Path:
    """Path of the daily .npz file that holds the crops of that UTC day."""
    d = utc(when)
    return RAW / "radar_crops" / f"{d:%Y}" / f"{d:%m}" / f"{d:%Y%m%d}.npz"


def crop_key(iata: str, when) -> str:
    """Key of one crop inside the .npz file, e.g. ATL_2025071520 (UTC hour)."""
    return f"{iata}_{utc(when):%Y%m%d%H}"


def load_crop(iata: str, when):
    """Stored crop of one airport at one UTC hour; None if that key is not in the file."""
    with np.load(crop_file(when)) as z:          # z.files = the list of keys
        key = crop_key(iata, when)
        return z[key] if key in z.files else None


mine = cut_crop(national, atl_row, atl_col, CROP_PX)
stored = load_crop("ATL", NAT_TIME)
print("file:", crop_file(NAT_TIME), "| key:", crop_key("ATL", NAT_TIME))
print("stored crop:", stored.shape, stored.dtype, "| my crop:", mine.shape, mine.dtype)
print("crop identical to the stored one:", np.array_equal(mine, stored))

**What:** the same comparison for **every** national image of the day and **all 10 hubs**. We also note the largest pixel value seen in the national images,
and count the pixels above 245 (the top of the documented scale, section B3) and the pixels equal to 255.
**Why:** one identical crop could be luck; all of them cannot. The counts tell us whether 255 ever appears as real data: it should not, because we reserve 255 for "outside the image".

In [ ]:
n_same = n_total = 0
nat_max_value = n_above_scale = n_255 = 0
for png in pngs:
    when = pd.to_datetime(png.stem.split("_")[1], format="%Y%m%d%H%M", utc=True)
    image, _palette = read_national(png)
    nat_max_value = max(nat_max_value, int(image.max()))
    n_above_scale += int((image > 245).sum())
    n_255 += int((image == OUTSIDE).sum())
    with np.load(crop_file(when)) as z:
        for hub in HUBS:
            row, col = latlon_to_pixel(AIRPORTS[hub]["lat"], AIRPORTS[hub]["lon"], wld)
            same = np.array_equal(cut_crop(image, row, col, CROP_PX), z[crop_key(hub, when)])
            n_same += int(same)
            n_total += 1
print(f"identical crops: {n_same} / {n_total} ({len(pngs)} hours x {len(HUBS)} hubs)")
print(f"national images of that day: {len(pngs) * n_rows * n_cols:,} pixels | largest value: "
      f"{nat_max_value} | pixels above 245: {n_above_scale} | pixels equal to 255: {n_255}")

**What:** the ATL crop with a marker at its centre.
**Why:** the airport must be exactly in the middle. The colours are IEM's own palette (the scale is explained in B3).

In [ ]:
CENTRE = CROP_PX // 2      # 192: the airport's row and column inside the crop


def show_crop(ax, crop: np.ndarray, title: str):
    """Draw one crop with IEM's colours and a white cross on the airport."""
    im = ax.imshow(crop, cmap=RADAR_CMAP, vmin=0, vmax=255, interpolation="nearest")
    ax.plot(CENTRE, CENTRE, marker="+", color="white", markersize=14, markeredgewidth=2)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    return im


fig, ax = plt.subplots(figsize=(5.5, 5.5))
show_crop(ax, stored, f"ATL crop, {NAT_TIME:%Y-%m-%d %H:%M} UTC\n"
                      f"white cross = pixel ({CENTRE}, {CENTRE}) = the airport")
plt.tight_layout()
plt.show()

**What:** how many kilometres the crop covers.
- North–south: 384 pixels × 0.005° = 1.92° of latitude, and one degree of latitude is about 111 km everywhere.
- East–west: also 1.92°, but of **longitude**. Meridians get closer towards the poles, so one degree of longitude is only `111 km × cos(latitude)`.

**Why:** the crop is a square in **pixels**, not in **kilometres**, and its width depends on the airport's latitude. The model sees a slightly different area at each hub.

In [ ]:
KM_PER_DEG = 111.2        # one degree of latitude in km (Earth radius 6,371 km x pi / 180)
crop_deg = CROP_PX * wld[0]

coverage = pd.DataFrame({"latitude": {hub: AIRPORTS[hub]["lat"] for hub in HUBS}})
coverage["north-south (km)"] = crop_deg * KM_PER_DEG
coverage["east-west (km)"] = crop_deg * KM_PER_DEG * np.cos(np.radians(coverage["latitude"]))
print(f"the crop covers {crop_deg:.2f} degrees in each direction")
print(f"at ATL: {coverage.loc['ATL', 'north-south (km)']:.0f} km north-south, "
      f"{coverage.loc['ATL', 'east-west (km)']:.0f} km east-west")
coverage.round(1).sort_values("latitude")

**Question:** At which pixel of the crop is the airport, and how far (in km) is the edge of the crop from it at ATL? A storm moves at about 40-50 km/h: can a storm that reaches the airport 2 hours after `T_pred` already be inside the crop at `T_pred`? Which hub has the narrowest crop in km, and why?

**My answer:**

## B3. From pixel value to storm intensity (dBZ)

**What:** radar intensity is measured in **dBZ** ("decibels of reflectivity"): the more and the bigger the drops, the higher the number.
For the `n0q` product, the pixel value (0–255) is converted with:

**`dBZ = value × 0.5 − 32.5`**

Two values are **not** intensities and must be set aside **before** converting:
- **0 = no echo** (the radar saw nothing: no rain). It is not "−32.5 dBZ".
- **255 = outside the national image** (our own padding, section B2). It is not weather at all.

`crop_to_dbz(arr)` returns decimal numbers with **NaN** for both.

**Check against the documentation** (IEM page linked in CLAUDE.md §4, https://mesonet.agron.iastate.edu/docs/nexrad_mosaic/, read on 2026-11-11):
the page describes the `n0q` colour ramp as "NODATA", then **−32 dBZ to 90 dBZ in 0.5 dBZ steps**. It does not write the formula itself;
the formula is what these three facts give: value 0 = no data, value 1 = −32 dBZ, each next value = +0.5 dBZ (so value 245 = 90 dBZ).
A second, independent check is in the image file: its palette shows the usual radar colours at the right places (next cell).

**Why:** the model will receive these values. If 255 or 0 were converted like real data, "no data" would become "extreme storm" or "very weak rain".

In [ ]:
NO_ECHO = 0       # the radar saw nothing


def crop_to_dbz(arr: np.ndarray) -> np.ndarray:
    """Pixel values -> dBZ (decimal numbers). NaN where there is nothing to convert:
    255 = outside the national image, 0 = no echo."""
    dbz = arr.astype(np.float32) * 0.5 - 32.5
    dbz[(arr == OUTSIDE) | (arr == NO_ECHO)] = np.nan
    return dbz


def share_at_least(arr: np.ndarray, dbz_min: float = 40.0) -> float:
    """% of the crop's pixels (those inside the national image) with at least `dbz_min` dBZ."""
    inside = arr != OUTSIDE
    if not inside.any():
        return NAN
    strong = crop_to_dbz(arr) >= dbz_min        # a NaN is never >= something: counts as False
    return float(100 * strong.sum() / inside.sum())


for value in [1, 105, 145, 165, 245, 255]:
    print(f"value {value:>3} -> {value * 0.5 - 32.5:>5.1f} dBZ | palette colour (R, G, B):",
          tuple(int(x) for x in palette[value]))
print("\ncrop_to_dbz on [0, 1, 145, 255]:", crop_to_dbz(np.array([0, 1, 145, 255], dtype=np.uint8)))

**What:** the palette of the image file drawn as a colour bar, with the dBZ scale given by the formula.
**Why:** every weather-radar map uses the same convention: green = light rain, yellow = heavy, red = very heavy. If the formula were wrong,
yellow and red would not fall on 40 and 50 dBZ.

With the formula, 255 would be **95.0 dBZ**, above the top of the documented scale (90 dBZ), and far above anything in real weather: reserving it for "outside" loses nothing.
The few pixels above ~75 dBZ counted in B2 are isolated single pixels: radar **artefacts** (a wrong measurement), not storms.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 1.5))
ax.imshow(np.arange(256)[None, :], cmap=RADAR_CMAP, vmin=0, vmax=255, aspect="auto")
ticks_dbz = [-30, -20, -10, 0, 10, 20, 30, 40, 50, 60, 70, 80, 90]
ax.set_xticks([(d + 32.5) * 2 for d in ticks_dbz], ticks_dbz)     # dBZ -> pixel value
ax.set_yticks([])
ax.set_xlabel("dBZ = value x 0.5 - 32.5   (value 0 on the far left = no echo, drawn black)")
ax.set_title("The palette stored in the PNG file")
plt.tight_layout()
plt.show()

### What the numbers mean

| dBZ | What it usually is |
|---|---|
| below ~15–20 | Almost nothing for us: drizzle, very light rain, or echoes that are not rain at all (insects, dust, ground clutter near the radar) |
| **20–30** | **Light rain** |
| 30–40 | Moderate rain |
| **40–50** | **Heavy rain**; often a shower or the edge of a thunderstorm |
| **50 and more** | **Very heavy rain: the core of a thunderstorm.** Above ~60, hail is likely |

For flights, the strong echoes (≥ 40 dBZ) are the ones that matter: aircraft fly around them, and airports slow down or stop when they are overhead.

**What:** histogram of the dBZ values of the ATL crop of B2 (pixels with an echo only), and the share of the crop above 20 / 40 / 50 dBZ.
**Why:** to see how a crop is filled: mostly "no echo", some light rain, and very few strong pixels even with a storm nearby.

In [ ]:
dbz = crop_to_dbz(stored)
echo = dbz[~np.isnan(dbz)]                 # keep the pixels that have a value
print(f"pixels in the crop: {stored.size:,}")
print(f"  no echo (0):          {100 * (stored == NO_ECHO).mean():.1f} %")
print(f"  outside image (255):  {100 * (stored == OUTSIDE).mean():.1f} %")
for level in [20, 40, 50]:
    print(f"  >= {level} dBZ:            {share_at_least(stored, level):.2f} %")
print(f"  strongest echo: {echo.max():.1f} dBZ")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(echo, bins=np.arange(-35, 80, 2.5), color="steelblue")
for level, style in zip([20, 40, 50], [":", "--", "-"], strict=True):
    ax.axvline(level, color="black", linestyle=style, linewidth=1.2, label=f"{level} dBZ")
ax.set_xlabel("dBZ (pixels with an echo)")
ax.set_ylabel("Number of pixels")
ax.set_title(f"ATL crop, {NAT_TIME:%Y-%m-%d %H:%M} UTC: distribution of echo intensity")
ax.legend()
plt.tight_layout()
plt.show()

**Question:** Why would it be dangerous to fill the part of a crop outside the image with 0 instead of 255? Why does `crop_to_dbz` give NaN for "no echo" instead of −32.5? In the histogram, what share of the crop is at 40 dBZ or more, and is that a lot?

**My answer:**

## B4. Storm day vs. calm day at ATL

**What:** for `STORM_DAY` and `CALM_DAY` (chosen from the METAR in A4), one line per UTC hour with:
- from the **radar** crop of that hour: the share of pixels ≥ 40 dBZ, and the strongest echo within about 11 km of the airport (20 pixels on each side);
- from the **METAR**: the last report **at or before** the image time (the same "at or before" rule as for `T_pred`), its thunderstorm flags and its raw text.

**Why:** two independent sources should tell the same story. If the radar shows strong echoes when the METAR reports thunderstorms, and almost nothing on the calm day,
then the crops, the dBZ conversion and the time alignment (both in UTC) are right.

In [ ]:
NEAR_PX = 20      # 20 px x 0.005 deg = 0.1 deg, about 11 km around the airport


def load_day_crops(iata: str, day: str) -> dict[int, np.ndarray]:
    """{UTC hour: crop} of one airport for one day (hours without a crop are left out)."""
    start = utc(day)
    crops = {}
    with np.load(crop_file(start)) as z:
        for hour in range(24):
            key = crop_key(iata, start + pd.Timedelta(hours=hour))
            if key in z.files:
                crops[hour] = z[key]            # reads this one array from the file
    return crops


def radar_vs_metar(iata: str, day: str) -> tuple[pd.DataFrame, dict[int, np.ndarray]]:
    """One line per UTC hour of `day`: radar numbers + the last METAR at or before the image."""
    start = utc(day)
    # The day before is read too: the image of 00:00 needs the last report of the evening before.
    reports = with_features(read_metar(iata, start - pd.Timedelta(days=1),
                                       start + pd.Timedelta(days=1)))
    crops = load_day_crops(iata, day)
    rows = []
    for hour, crop in crops.items():
        when = start + pd.Timedelta(hours=hour)
        report = last_report_at_or_before(reports, when)
        near = crop_to_dbz(crop[CENTRE - NEAR_PX:CENTRE + NEAR_PX,
                                CENTRE - NEAR_PX:CENTRE + NEAR_PX])
        rows.append({
            "hour": hour,
            "pct_40dbz": share_at_least(crop, 40),
            "max_dbz_near": float(np.nanmax(near)) if not np.isnan(near).all() else NAN,
            "metar_time": f"{report['valid']:%m-%d %H:%M}",
            "ts_in_weather": report["ts_in_weather"],
            "thunderstorm": report["thunderstorm"],
            "metar": report["metar"],
        })
    return pd.DataFrame(rows).set_index("hour"), crops


storm_table, storm_crops = radar_vs_metar("ATL", STORM_DAY)
calm_table, calm_crops = radar_vs_metar("ATL", CALM_DAY)
print(f"crops found: {len(storm_crops)} hours on {STORM_DAY}, {len(calm_crops)} on {CALM_DAY}")
print("share of crop pixels >= 40 dBZ, mean over the day / highest hour:")
print(f"  STORM_DAY {STORM_DAY}: {storm_table['pct_40dbz'].mean():.2f} % "
      f"/ {storm_table['pct_40dbz'].max():.2f} %")
print(f"  CALM_DAY  {CALM_DAY}: {calm_table['pct_40dbz'].mean():.2f} % "
      f"/ {calm_table['pct_40dbz'].max():.2f} %")

**What:** the crops of four hours around the departure of A6 (from `T_PRED` to one hour after departure), for the storm day (top row) and the same hours of the calm day (bottom row).
All eight pictures use the **same colour scale**. Under each title: the time of the last METAR before the image and whether it reports `TS`.
**Why:** look at the storm cells moving from picture to picture, and at where they are relative to the white cross (the airport).

In [ ]:
SHOW_HOURS = [h for h in range(DEP_UTC.hour - 2, DEP_UTC.hour + 2) if 0 <= h <= 23]

fig, axes = plt.subplots(2, len(SHOW_HOURS), figsize=(3.6 * len(SHOW_HOURS), 7.6),
                         layout="constrained")
cases = [(STORM_DAY, storm_table, storm_crops), (CALM_DAY, calm_table, calm_crops)]
for axes_row, (day_text, table, crops) in zip(axes, cases, strict=True):
    for ax, hour in zip(axes_row, SHOW_HOURS, strict=True):
        line = table.loc[hour]
        ts_text = "TS" if line["ts_in_weather"] else "no TS"
        im = show_crop(ax, crops[hour], f"{day_text} {hour:02d}:00 UTC\n"
                                        f"METAR {line['metar_time'][-5:]}: {ts_text}")
# One colour bar for all pictures, labelled in dBZ.
bar = fig.colorbar(im, ax=axes, shrink=0.7)
bar_dbz = [0, 10, 20, 30, 40, 50, 60, 70]
bar.set_ticks([(d + 32.5) * 2 for d in bar_dbz], labels=bar_dbz)
bar.set_label("dBZ")
fig.suptitle("ATL radar crops: storm day (top) and calm day (bottom)")
plt.show()

In [ ]:
# The same hours as a table: radar numbers next to the METAR text.
shown = pd.concat({STORM_DAY: storm_table.loc[SHOW_HOURS], CALM_DAY: calm_table.loc[SHOW_HOURS]})
shown.round(2)

**What:** for every hour of both days: the share of crop pixels ≥ 40 dBZ (bars) and the METAR thunderstorm flags (marks above the bars).
- red triangle = `TS` in a weather group (thunderstorm at the airport);
- orange circle = a thunderstorm hint **only in the remarks** (distant, or just ended).

**Why:** the eight pictures show a few hours; this chart shows whether radar and METAR agree over the whole two days.

In [ ]:
top = max(storm_table["pct_40dbz"].max(), calm_table["pct_40dbz"].max())
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, (day_text, table, _crops) in zip(axes, cases, strict=True):
    ax.bar(table.index, table["pct_40dbz"], color="steelblue", label="% of crop >= 40 dBZ")
    ts_now = table["ts_in_weather"].eq(True)
    hint_only = table["thunderstorm"].eq(True) & ~ts_now
    ax.scatter(table.index[ts_now], [top * 1.12] * int(ts_now.sum()), marker="v", color="red",
               s=60, label="METAR: TS in a weather group")
    ax.scatter(table.index[hint_only], [top * 1.05] * int(hint_only.sum()), marker="o",
               facecolors="none", edgecolors="darkorange", s=40,
               label="METAR: hint in the remarks only")
    ax.set_xticks(range(0, 24, 2))
    ax.set_ylim(0, top * 1.2)
    ax.set_xlabel("Hour of the image (UTC)")
    ax.set_title(f"ATL, {day_text}")
axes[0].set_ylabel("% of crop pixels >= 40 dBZ")
# One legend for both charts, under them (inside a chart it would hide the marks).
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", ncol=3, fontsize=9)
plt.tight_layout(rect=(0, 0.08, 1, 1))       # leave the bottom 8 % of the figure for the legend
plt.show()

**Question:** Do the strong echoes match the METAR thunderstorm reports? Find one hour where the radar shows strong echoes in the crop but the METAR has no `TS` in a weather group: where is the storm relative to the airport (use `max_dbz_near` and the pictures)? In general, when can the radar show a storm that the METAR does not report, and when can the METAR report a thunderstorm that the radar hardly shows?

**My answer:**

## B5. Gaps: what is missing in the radar data

**What:** the two lists written by `download_radar`:
- `missing_files.txt`: national images that **do not exist on the IEM server** (the radar or the archive failed at that time); downloading again cannot help;
- `skipped_airports.txt`: airports **outside the image** (the composite covers the contiguous US only).

**Why:** Golden rule 5: no silent drops. Some flights will have **no radar image**. We must know how many and why, and the model must be able to work without the image.

In [ ]:
def read_lines(path: Path) -> list[str]:
    """The non-empty lines of a small text file."""
    return [ln.strip() for ln in path.read_text(encoding="utf-8").splitlines() if ln.strip()]


missing_files = read_lines(RAW / "radar_crops" / "missing_files.txt")
skipped = read_lines(RAW / "radar_crops" / "skipped_airports.txt")
missing_pngs = [name for name in missing_files if name.endswith(".png")]
other_lines = [name for name in missing_files if not name.endswith(".png")]
print(f"missing hourly images: {len(missing_pngs)} | other lines in the file: {other_lines}")
print(f"airports without radar: {len(skipped)}")

# 'n0q_202604012000.png' -> 2026-04-01 20:00 UTC
missing_hours = pd.Series(pd.to_datetime(
    [name.split("_")[1][:12] for name in missing_pngs], format="%Y%m%d%H%M", utc=True))
missing_hours = missing_hours.sort_values(ignore_index=True)


def split_of(when: pd.Timestamp) -> str:
    """train / validation / test: the split (config.yaml) that contains this moment."""
    for name, (first_day, last_day) in cfg["split"].items():
        if utc(first_day) <= when < utc(last_day) + pd.Timedelta(days=1):
            return name
    return "outside the splits"


# Group consecutive hours: a new block starts when the step from the hour before is not 1 hour.
block = (missing_hours.diff() != pd.Timedelta(hours=1)).cumsum()
blocks = missing_hours.groupby(block).agg(first_hour="min", last_hour="max", hours="size")
blocks["split"] = blocks["first_hour"].map(split_of)
blocks

**What:** the airports without radar, with their state or territory, and their position in pixels.
**Why:** to confirm the reason: their row or column falls outside the image (negative, or larger than the image size).

In [ ]:
rows = []
for code in skipped:
    info = AIRPORTS[code]
    row, col = latlon_to_pixel(info["lat"], info["lon"], wld)
    rows.append({"iata": code, "name": info["name"], "state / territory": info["subd"],
                 "country": info["country"], "row": row, "col": col,
                 "inside the image": 0 <= row < n_rows and 0 <= col < n_cols})
no_radar = pd.DataFrame(rows).set_index("iata")
print("airports per state / territory:")
print(no_radar["state / territory"].value_counts().to_string())
no_radar

**What:** for one month, compare what the flights **need** with what is **on disk**.
`radar_needed_hours.parquet` lists every (airport, UTC hour) pair needed: the hubs at every hour, and the destinations at `T_pred` and the 2 hours before.
We build the key of each pair and look for it in the daily `.npz` files (only the list of keys is read, not the images). Each absent key gets a reason.

Run for July 2025 (the month of this notebook) and for April 2026 (the month of the missing images).

**Why:** this is the radar **match rate** for the data-quality report. "Unexplained" must be 0: every absent crop must have a known reason.

In [ ]:
missing_hour_texts = set(missing_hours.dt.strftime("%Y%m%d%H"))


def absent_crops(month_start: str) -> pd.DataFrame:
    """Needed (airport, hour) pairs of one month that have no crop on disk, with the reason."""
    start = utc(month_start)
    end = start + pd.offsets.MonthBegin(1)               # first day of the next month
    # filters = read only the rows of this month from the Parquet file, not the whole file
    needed = pd.read_parquet(RAW / "radar_needed_hours.parquet",
                             filters=[("hour_utc", ">=", start), ("hour_utc", "<", end)])
    hour_text = needed["hour_utc"].dt.strftime("%Y%m%d%H")
    needed["key"] = needed["airport"] + "_" + hour_text

    have: set[str] = set()                               # every key present in the month's files
    for d in pd.date_range(start, end, freq="D", inclusive="left"):
        if crop_file(d).exists():
            with np.load(crop_file(d)) as z:
                have.update(z.files)

    absent = needed[~needed["key"].isin(have)].copy()
    absent["reason"] = "UNEXPLAINED"
    absent.loc[hour_text.isin(missing_hour_texts), "reason"] = "image missing on the server"
    absent.loc[absent["airport"].isin(skipped), "reason"] = "airport outside the image"
    print(f"{month_start[:7]}: needed {len(needed):,} | on disk {len(needed) - len(absent):,} "
          f"({100 * (1 - len(absent) / len(needed)):.2f} %) | absent {len(absent):,}")
    return absent


for month in ["2025-07-01", "2026-04-01"]:
    absent = absent_crops(month)
    print(absent["reason"].value_counts().to_string(), "\n")

**What:** does the padding value **255** appear in the crops?
1. **By geometry:** for every airport of `stations.csv`, the distance in pixels from the airport to the nearest border of the image. A crop reaches 192 pixels on each side, so padding is needed only if that distance is below 192.
2. **By measurement:** count the 255 pixels in the stored crops of one day, for an airport far from the border (ATL), for SEA (in the north-west corner of the country), and for the covered airport closest to a border; then in **all** crops of that day.

**Why:** 255 must appear only where a crop really sticks out of the image, and never inside real data. If it never appears at all, the padding rule is a safety net.
The mask in `crop_to_dbz` stays anyway: it costs nothing, and it also removes a possible artefact pixel that would reach 255 (B2 shows that isolated pixels can go above the documented scale).

In [ ]:
stations = pd.read_csv(RAW / "stations.csv")
pixels = [latlon_to_pixel(lat, lon, wld)
          for lat, lon in zip(stations["lat"], stations["lon"], strict=True)]
stations["row"] = [row for row, _col in pixels]
stations["col"] = [col for _row, col in pixels]
# Distance to the nearest of the 4 borders (negative = the airport itself is outside).
stations["px_to_border"] = np.minimum.reduce([
    stations["row"], n_rows - 1 - stations["row"],
    stations["col"], n_cols - 1 - stations["col"]])

covered = stations[stations["px_to_border"] >= 0].sort_values("px_to_border")
need_padding = covered[covered["px_to_border"] < CENTRE]
print(f"airports in stations.csv: {len(stations)} | inside the image: {len(covered)} "
      f"| outside: {len(stations) - len(covered)}")
print(f"covered airports closer than {CENTRE} px to a border (crop needs padding): "
      f"{len(need_padding)}")
covered[["iata", "lat", "lon", "row", "col", "px_to_border"]].head(5)

In [ ]:
closest = covered["iata"].iloc[0]          # the covered airport nearest to a border
with np.load(crop_file(NAT_DAY)) as z:
    for code in ["ATL", "SEA", closest]:
        keys = [k for k in z.files if k.startswith(code + "_")]
        n_255 = sum(int((z[k] == OUTSIDE).sum()) for k in keys)
        n_px = len(keys) * CROP_PX * CROP_PX
        print(f"{code}: {len(keys)} crops on {NAT_DAY:%Y-%m-%d} | pixels = 255: {n_255:,} "
              f"({100 * n_255 / n_px:.3f} %)")
    # All crops of that day (a few seconds: every array of the file is read once).
    n_255_all = sum(int((z[k] == OUTSIDE).sum()) for k in z.files)
    print(f"all {len(z.files):,} crops of that day: pixels = 255: {n_255_all:,}")

### How the model will have to handle a missing image (for Phase 2 and the data-quality report)

A flight can lack a radar image for three reasons, all counted above:
1. its **destination is outside the composite** (Alaska, Hawaii, Puerto Rico, US Virgin Islands): never any image;
2. the **national image does not exist** for that hour on the server (one block of hours in April 2026, in the **test** period);
3. (later) a crop not downloaded yet for a new month.

Rules:
- **A flag, not a guess.** Gold gets `has_radar_origin` and `has_radar_dest` (and the same per frame if several frames are used). A missing image is given to the model as an
  empty image **plus** the flag at 0, so the model can learn "no information", which is different from "clear sky" (an image full of 0).
- **Never a future frame.** If the frame at `T_pred` is missing, the only allowed replacement is an **earlier** frame (≤ `T_pred`), with its age stored as a feature. Taking the next frame would be leakage.
- **No silent drop.** Flights without an image stay in the dataset (the other modalities still work); the report gives the radar match rate per split and per reason.
- **Fit on train only.** Any statistic used to normalise images is computed on training crops, ignoring 255 and missing images.
- The same idea applies to a missing TAF (A7): `has_taf`.

**Question:** How many hourly images are missing, on which dates, and in which split (train / validation / test) do they fall? Which airports have no radar at all, and why? If the frame at `T_pred` is missing, why is it forbidden to use the next one, and what are the two honest options?

**My answer:**

---
## B6. Summary of Part B

What this part contains, to reuse in `src/transform/crop_radar.py`, the Gold table and the tests:
- **Grid:** the national `n0q` image is a regular longitude / latitude grid described by 6 numbers (`.wld`); `latlon_to_pixel` finds an airport in it.
  The grid and the image size match `config.yaml`.
- **Crops:** a crop rebuilt by hand is identical to the stored one, for every hub and every hour of the debug day. The airport is at pixel (192, 192).
  The crop is a square in pixels, not in km: its east–west width shrinks with latitude.
- **Values:** `dBZ = value × 0.5 − 32.5`; 0 = no echo and 255 = outside the image are masked **before** converting (`crop_to_dbz`). Strong echoes = 40 dBZ and more.
- **Storm vs. calm:** the radar of the METAR-chosen storm day and calm day, hour by hour, next to the thunderstorm flags.
- **Gaps:** missing national images and airports outside the composite are listed, counted and explained; nothing absent is unexplained.
  The model needs a `has_radar` flag and must never replace a missing frame by a later one.

**Limits to remember:** one debug day of national images (the identity check covers that day only); July 2025 for the statistics (winter weather looks different on radar: snow gives weaker echoes);
the share of pixels ≥ 40 dBZ is computed on the whole crop, so a storm 80 km away counts as much as one over the runway;
a few isolated pixels have impossible values (above ~75 dBZ): cap the dBZ values (for example at 75) before giving crops to a model.

### Check questions of steps 1.3 and 1.4 (CLAUDE.md): answer without notes

**Question:** What is the difference between a METAR and a TAF? (Who writes it, what moment does it describe, how often does it come?)

**My answer:**

**Question:** Why must a prediction made 2 h before departure use the TAF, and not a METAR from the departure time? Which METAR is it allowed to use?

**My answer:**

**Question:** Does a known storm day show strong echoes on the radar, and a clear day almost nothing? Give the numbers you saw in B4 (share of pixels ≥ 40 dBZ on each day).

**My answer:**

---
*End of notebook 02.*